# Fitting Models


Figures are from Prince, *[Understanding Deep Learning](https://udlbook.github.io/udlbook/)*
(UDL, chapter 6), unless another source is cited. Code cells adapted from the
UDL notebooks (MIT license,
[github.com/udlbook/udlbook/tree/main/Notebooks](https://github.com/udlbook/udlbook/tree/main/Notebooks)).

Last lecture turned a model and a dataset into a single number, the loss
$L[\boldsymbol{\phi}]$. This lecture is about driving that number down: plain
gradient descent, which you have seen in DS701 or DS340, and then the three
additions that every modern training run uses on top of it: stochastic
mini-batches, momentum and per-parameter step sizes (Adam). We finish with two
pieces that the textbook treats lightly but every 2026 language-model recipe
depends on: learning-rate schedules and decoupled weight decay (AdamW).


## Loss function and training

- Training dataset of $I$ input/output pairs $\{\mathbf{x}_i, \mathbf{y}_i\}_{i=1}^{I}$
- A **loss function** returns a scalar that is smaller when the model maps inputs
  to outputs better:

$$
L\left[\boldsymbol{\phi}, \mathrm{f}[\mathbf{x}_i, \boldsymbol{\phi}], \{\mathbf{x}_i, \mathbf{y}_i\}_{i=1}^{I}\right]
\quad\text{or for short}\quad L[\boldsymbol{\phi}]
$$

- **Training** means finding the parameters that minimize it:

$$
\hat{\boldsymbol{\phi}} = \underset{\boldsymbol{\phi}}{\mathrm{argmin}}\left[L[\boldsymbol{\phi}]\right]
$$


- The data are fixed; the **parameters** $\boldsymbol{\phi}$ are the variables we move.


## Outline

- Gradients and the gradient descent algorithm
  - Linear regression example (convex)
  - Gabor model example (non-convex)
- Stochastic gradient descent: batches and epochs
- Momentum and Nesterov momentum
- Adam: per-parameter step sizes with bias correction
- Learning-rate schedules: warmup, cosine decay, warmup-stable-decay
- Weight decay vs. L2 regularization: AdamW
- What a 2026 LLM recipe uses, and Muon

# Gradients and gradient descent

Both DS701 and DS340 covered gradient descent, so this section moves quickly:
one slide to fix what a derivative tells us, one for the gradient of a loss over
many parameters, and one for the algorithm itself.


## Derivative and slope


![](figs/04-Fitting-Models/parabola-tangent-5.png)


$$
y = x^2 - 4x + 5, \qquad \frac{\partial y}{\partial x} = 2x - 4
$$

The derivative is the slope $m$ of the tangent line at the point where we evaluate it.

- $x = 2$: $m = 0$, the minimum.


- $x = 3$: $m = 2$. Slope $> 0$, so move in the $-x$ direction.


- $x = 4$: $m = 4$. Steeper, further from the minimum.


- $x = 1$: $m = -2$. Slope $< 0$, so move in the $+x$ direction.


- $x = 0$: $m = -4$. Moving **against** the sign of the slope always moves
  toward the minimum.


Definitions, for reference:

- The **derivative** quantifies how sensitive a function's output is to a
  change in its input. A function is *differentiable* at a point $a$ if the
  limit $\displaystyle\lim_{h \to 0} \frac{f(a+h) - f(a)}{h}$ exists; that
  limit is the derivative at $a$, and a small $h$ approximates it.
- The **gradient** of a function of several variables is the vector of its
  partial derivatives, one per variable; it gives the degree and direction of
  steepest ascent at a point.


## Gradient of the loss

The **gradient** collects the partial derivative with respect to each parameter:

$$
\frac{\partial L}{\partial \boldsymbol{\phi}} = \begin{bmatrix}
\frac{\partial L}{\partial \phi_0} \\[4pt]
\frac{\partial L}{\partial \phi_1} \\
\vdots \\
\frac{\partial L}{\partial \phi_N}
\end{bmatrix},
\qquad\text{also written } \nabla_{\boldsymbol{\phi}} L.
$$

Each entry is the rate of change of the loss when only that parameter moves.


![](figs/04-Fitting-Models/gradient-3d-surface.png)

Geometric interpretation: the gradient is the vector sum of the per-parameter
slopes, and points in the direction of **greatest increase** of the loss.


## Gradient descent algorithm

**Step 1.** Compute the gradient of the loss with respect to the parameters,
$\partial L / \partial \boldsymbol{\phi}$, at the current parameters.

**Step 2.** Move a small distance in the opposite direction:

$$
\boldsymbol{\phi} \longleftarrow \boldsymbol{\phi} - \alpha \frac{\partial L}{\partial \boldsymbol{\phi}},
$$

where the positive scalar $\alpha$ is the **step size**, or **learning rate**
when it is fixed.

Repeat until the loss stops decreasing.


Nothing in these two steps is specific to neural networks: the model only has
to be differentiable in its parameters. Where the gradient *comes from* for a
deep network (backpropagation) is the subject of the next lecture; today we
treat it as given.


# Linear regression example

We apply the two steps to the 1D linear regression model, whose loss over
$\phi_0$ and $\phi_1$ we can draw as a contour plot. This is the one case where
we work out the gradient by hand.


## Least squares loss

![](figs/04-Fitting-Models/linear-regression-loss.png)


Model and loss:

$$
\mathrm{f}[x, \boldsymbol{\phi}] = \phi_0 + \phi_1 x
$$

$$
\begin{aligned}
L[\boldsymbol{\phi}] &= \sum_{i=1}^{I} \left(\mathrm{f}[x_i, \boldsymbol{\phi}] - y_i\right)^2 \\
&= \sum_{i=1}^{I} \left(\phi_0 + \phi_1 x_i - y_i\right)^2
\end{aligned}
$$

The **least squares** loss: the sum of the squared vertical distances.


## Step 1: the gradient

![](figs/04-Fitting-Models/gd-loss-start.png)


Write the loss as a sum of per-example losses $\ell_i$:

$$
L[\boldsymbol{\phi}] = \sum_{i=1}^{I} \ell_i, \qquad \ell_i = \left(\phi_0 + \phi_1 x_i - y_i\right)^2
$$

The derivative of a sum is the sum of the derivatives, so we **accumulate** the
gradient evaluated at each data point:

$$
\frac{\partial L}{\partial \boldsymbol{\phi}} = \sum_{i=1}^{I} \frac{\partial \ell_i}{\partial \boldsymbol{\phi}}
$$


$$
\frac{\partial \ell_i}{\partial \boldsymbol{\phi}} = \begin{bmatrix}
\frac{\partial \ell_i}{\partial \phi_0} \\[4pt]
\frac{\partial \ell_i}{\partial \phi_1}
\end{bmatrix} = \begin{bmatrix}
2(\phi_0 + \phi_1 x_i - y_i) \\
2x_i(\phi_0 + \phi_1 x_i - y_i)
\end{bmatrix}
$$


## Step 2: the update


![](figs/04-Fitting-Models/gd-loss-gradient-components.png)

![](figs/04-Fitting-Models/gd-loss-step-line.png)


At point 0 the two components $\partial L/\partial \phi_0$ and
$\partial L/\partial \phi_1$ add to a gradient vector that points uphill.

Step 2 moves the opposite way:

$$
\boldsymbol{\phi} \longleftarrow \boldsymbol{\phi} - \alpha \frac{\partial L}{\partial \boldsymbol{\phi}}
$$

How far we move along that line is set by $\alpha$.


## Iterate


Repeating the two steps from each new point moves the parameters downhill
until the fitted line matches the data. The cell below runs both steps on the
12-point dataset of UDL figure 6.1. Like that figure, it picks each step size
by **line search**: it moves along $-\partial L/\partial\boldsymbol{\phi}$ to
the lowest point on that line. For a quadratic loss that step has a closed
form, $\alpha = \mathbf{g}^\top\mathbf{g} / \mathbf{g}^\top\mathbf{H}\mathbf{g}$,
where $\mathbf{H}$ is the (constant) Hessian.

In [ ]:
#| code-fold: true
#| fig-cap: "Gradient descent with line search on the linear regression loss, from $\\boldsymbol{\\phi} = (1.6, -0.5)$. Left: the path on the loss contours. Right: the line after steps 0 to 4 and after step 10."
#| fig-alt: "Left: contour plot of the linear regression loss over intercept phi_0 and slope phi_1 with a path of numbered points zigzagging from the start at (1.6, -0.5) toward the minimum near (1.0, 0.3). Right: the 12 training points with fitted lines after steps 0 to 4 and 10; the first line slopes down, later lines follow the upward trend of the data."
# Adapted from UDL notebook 6.2 (MIT)
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

udl_cmap = LinearSegmentedColormap.from_list(
    "udl", ["#2a0902", "#a35d4b", "#f2bd9f", "#ffffe0"])

x_lin = np.array([0.03, 0.19, 0.34, 0.46, 0.78, 0.81, 1.08, 1.18, 1.39, 1.60, 1.65, 1.90])
y_lin = np.array([0.67, 0.85, 1.05, 1.00, 1.40, 1.50, 1.30, 1.54, 1.55, 1.68, 1.73, 1.60])

def linreg_loss(phi):
    return np.sum((phi[0] + phi[1] * x_lin - y_lin) ** 2)

def linreg_grad(phi):
    r = phi[0] + phi[1] * x_lin - y_lin
    return np.array([np.sum(2 * r), np.sum(2 * x_lin * r)])

A = np.column_stack([np.ones_like(x_lin), x_lin])
H = 2 * A.T @ A                       # Hessian of the least squares loss

phi = np.array([1.6, -0.5])
lin_path = [phi]
for step in range(10):
    g = linreg_grad(phi)              # Step 1: the gradient
    alpha = (g @ g) / (g @ H @ g)     # line search: best step along -g
    phi = phi - alpha * g             # Step 2: the update
    lin_path.append(phi)
lin_path = np.array(lin_path)

for t in [0, 1, 2, 3, 4, 10]:
    print(f"step {t:2d}: phi = ({lin_path[t, 0]:.3f}, {lin_path[t, 1]:+.3f}), "
          f"loss = {linreg_loss(lin_path[t]):.4f}")

def plot_linreg_loss(ax, path):
    p0, p1 = np.meshgrid(np.linspace(0, 2, 201), np.linspace(-1, 1, 201))
    r = p0[..., None] + p1[..., None] * x_lin - y_lin
    loss = np.sum(r ** 2, axis=-1)
    ax.contourf(p0, p1, loss, 256, cmap=udl_cmap)
    ax.contour(p0, p1, loss, 40, colors="#80808080", linewidths=0.8)
    ax.plot(path[:, 0], path[:, 1], "o-", color="#1baf7a", ms=5)
    ax.set_xlabel(r"Intercept $\phi_0$")
    ax.set_ylabel(r"Slope $\phi_1$")

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(10, 4.5))
plot_linreg_loss(ax0, lin_path)
for t in range(5):
    ax0.annotate(str(t), lin_path[t], xytext=(6, 4), textcoords="offset points", color="white")
xs = np.linspace(0, 2, 2)
ax1.plot(x_lin, y_lin, "o", color="#eb6834", mec="#52514e")
for t, shade in zip([0, 1, 2, 3, 4, 10], np.linspace(0.25, 1.0, 6)):
    ax1.plot(xs, lin_path[t, 0] + lin_path[t, 1] * xs, color="#2a78d6", alpha=shade,
             label=f"step {t}")
ax1.set_xlim(0, 2); ax1.set_ylim(0, 2)
ax1.set_xlabel("Input $x$"); ax1.set_ylabel("Output $y$")
ax1.legend(fontsize=8, loc="lower right")
plt.tight_layout()
plt.show()

# Gabor model example

The linear regression loss is **convex**: a bowl with a single minimum, so
gradient descent from any start reaches the same answer. Neural network losses
are not. To see what gradient descent does on a non-convex loss, we switch to
a two-parameter model whose loss surface we can still plot.


## A non-convex loss: the Gabor model

The linear model's loss was **convex**: every local minimum is the global
minimum, so where we start does not matter.

Neural network losses are **non-convex**. To see what that does to gradient
descent, we use a two-parameter model we can still visualize (UDL eq. 6.8):

$$
\mathrm{f}[x, \boldsymbol{\phi}] = \sin[\phi_0 + 0.06 \cdot \phi_1 x] \cdot \exp\left(-\frac{(\phi_0 + 0.06 \cdot \phi_1 x)^2}{32.0}\right)
$$

A sinusoid inside a Gaussian envelope: a **Gabor function**.


In [ ]:
#| code-fold: true
#| fig-alt: "Plot over x from -15 to 15 of the 30 training points, the Gabor model with phi_0 = -5 and phi_1 = 25 as an oscillating solid curve, and its Gaussian envelope as a dashed curve peaking at 1 near x = 3.3."
#| out-width: 60%
# Adapted from UDL notebook 6.3 (MIT)
x_gab = np.array([-1.920, -14.22, 1.490, -1.940, -2.389, -5.090, -8.861, 3.578, -6.010, -6.995,
                  3.634, 0.8743, -10.96, 0.4073, -9.467, 8.560, 10.62, -0.1729, 10.40, -12.61,
                  0.1574, -13.04, -2.156, -12.10, -11.19, 2.902, -8.220, -11.79, -8.391, -4.505])
y_gab = np.array([-1.051, -0.02482, 0.8896, -0.4943, -0.9371, 0.4306, 0.009577, -0.07944,
                  0.1624, -0.2682, -0.3129, 0.8303, -0.02365, 0.5098, -0.2777, 0.3367, 0.1927,
                  -0.2222, 0.06352, 0.006888, 0.03224, 0.01091, -0.5706, -0.05258, -0.03666,
                  0.1709, -0.04805, 0.2008, -0.1904, 0.5952])

def gabor(phi, x):
    z = phi[0] + 0.06 * phi[1] * x
    return np.sin(z) * np.exp(-z ** 2 / 32.0)

phi = np.array([-5.0, 25.0])
xs = np.linspace(-15, 15, 600)
z = phi[0] + 0.06 * phi[1] * xs
fig, ax = plt.subplots(figsize=(8, 2.6))
ax.plot(x_gab, y_gab, "o", color="#eb6834", mec="#52514e", ms=4, label="training data")
ax.plot(xs, gabor(phi, xs), color="#2a78d6", label="model")
ax.plot(xs, np.exp(-z ** 2 / 32.0), "--", color="#1baf7a", label="envelope")
ax.set_xlabel("$x$"); ax.set_ylabel("$y$")
ax.set_title(r"$\phi_0 = -5,\ \phi_1 = 25$", fontsize=10)
ax.legend(fontsize=8, loc="lower left")
plt.tight_layout()
plt.show()

## Gabor model parameters

$$
\mathrm{f}[x, \boldsymbol{\phi}] = \sin[\phi_0 + 0.06 \cdot \phi_1 x] \cdot \exp\left(-\frac{(\phi_0 + 0.06 \cdot \phi_1 x)^2}{32.0}\right)
$$



- $\phi_0$ shifts the function left and right
- $\phi_1$ shrinks and expands the sinusoid and its envelope


Interactive version of UDL Figure 6.2, ported from the UDL interactive figures (MIT).


The interactive figure does not reach the notebook; the Gabor cell above
defines `gabor(phi, x)` and plots the same curve for
$\boldsymbol{\phi} = (-5, 25)$. Change `phi` there to try other settings.



## Toy dataset and loss

![](figs/04-Fitting-Models/gabor-toy-data-and-model.png)

Same least squares loss as before, with the Gabor model in place of the line:

$$
L[\boldsymbol{\phi}] = \sum_{i=1}^{I} \left(\mathrm{f}[x_i, \boldsymbol{\phi}] - y_i\right)^2
$$

## Gabor model loss surface



Interactive version of UDL Figure 6.4, ported from the UDL interactive figures (MIT).


The interactive figure does not reach the notebook; the gradient descent cell
below computes the same loss on a grid (`gabor_loss_grid`) and
plots it with `plot_gabor_loss`, and `gabor_loss(phi)` gives the loss at any
single point.


## Gradient descent on the Gabor model

![](figs/04-Fitting-Models/gabor-gradient-descent-paths.png)


- Reaches the **global minimum** only if it starts in the right "valley" (path 1)
- Otherwise descends to a **local minimum** (path 2)
- Or stalls near a **saddle point**, where the gradient is near zero but the
  point is a minimum in one direction and a maximum in another (path 3)

Gradient descent only sees the local slope. It cannot tell these three outcomes apart.


For a deep network with millions of parameters the picture is far higher
dimensional, and the local minima that gradient descent finds are in practice
good enough; it is the saddle points and the long, flat valleys that slow
training down. The optimizers in the rest of this lecture are mostly about
getting through those faster.

The cell below writes out the gradient of the Gabor loss, checks it against
finite differences, and runs fixed-step gradient descent from three starts.
Change the starts or $\alpha$ and watch which valley each path ends in.

In [ ]:
#| code-fold: true
#| fig-cap: "Fixed-step gradient descent ($\\alpha = 0.1$, 200 steps) on the Gabor loss from three starts. Only the start in the central valley reaches the global minimum."
#| fig-alt: "Contour plot of the Gabor loss over phi_0 from -10 to 10 and phi_1 from 2.5 to 22.5 with three green paths. Path A, from (-2, 20.3), drops into the central dark valley at the global minimum near (0, 16). Path B, from (2.5, 8), climbs right into a valley near (6.3, 13.5). Path C, from (-4, 12), settles in a valley near (-5.8, 13.8)."
# Adapted from UDL notebook 6.3 (MIT)
def gabor_loss(phi, x=x_gab, y=y_gab):
    return np.sum((gabor(phi, x) - y) ** 2)

def gabor_grad(phi, x=x_gab, y=y_gab):
    z = phi[0] + 0.06 * phi[1] * x
    envelope = np.exp(-z ** 2 / 32.0)
    df_dz = envelope * (np.cos(z) - np.sin(z) * z / 16.0)
    dL_dz = 2 * (np.sin(z) * envelope - y) * df_dz
    return np.array([np.sum(dL_dz), np.sum(dL_dz * 0.06 * x)])   # chain rule: dz/dphi

phi = np.array([-5.0, 25.0])
h = 1e-6
fd = [(gabor_loss(phi + h * e) - gabor_loss(phi - h * e)) / (2 * h) for e in np.eye(2)]
print("analytic gradient:", np.round(gabor_grad(phi), 5), " finite differences:", np.round(fd, 5))

P0, P1 = np.meshgrid(np.arange(-10, 10, 0.05), np.arange(2.5, 22.5, 0.05))
Z = P0[..., None] + 0.06 * P1[..., None] * x_gab
gabor_loss_grid = np.sum((np.sin(Z) * np.exp(-Z ** 2 / 32.0) - y_gab) ** 2, axis=-1)
i = np.unravel_index(gabor_loss_grid.argmin(), gabor_loss_grid.shape)
print(f"lowest loss on the grid: {gabor_loss_grid[i]:.3f} at phi = ({P0[i]:.2f}, {P1[i]:.2f})")

def plot_gabor_loss(ax, paths, title=None):
    ax.contourf(P0, P1, gabor_loss_grid, 256, cmap=udl_cmap)
    ax.contour(P0, P1, gabor_loss_grid, 20, colors="#80808080", linewidths=0.8)
    for path in paths:
        ax.plot(path[:, 0], path[:, 1], ".-", color="#1baf7a", ms=4, lw=1)
    ax.set_xlabel(r"Offset $\phi_0$"); ax.set_ylabel(r"Frequency $\phi_1$")
    if title:
        ax.set_title(title, fontsize=10)

def gradient_descent(start, alpha=0.1, n_steps=200):
    phi = np.array(start, dtype=float)
    path = [phi]
    for _ in range(n_steps):
        phi = phi - alpha * gabor_grad(phi)
        path.append(phi)
    return np.array(path)

starts = {"A": (-2.0, 20.3), "B": (2.5, 8.0), "C": (-4.0, 12.0)}
gd_paths = {name: gradient_descent(s) for name, s in starts.items()}
for name, path in gd_paths.items():
    print(f"start {name} {starts[name]}: ends at ({path[-1, 0]:.2f}, {path[-1, 1]:.2f}), "
          f"loss = {gabor_loss(path[-1]):.2f}")

fig, ax = plt.subplots(figsize=(5.5, 5.5))
plot_gabor_loss(ax, gd_paths.values())
for name, s in starts.items():
    ax.annotate(name, s, xytext=(-12, 4), textcoords="offset points", color="white", fontsize=12)
plt.show()

# Stochastic gradient descent

Where gradient descent ends up depends on where it starts. Stochastic gradient
descent adds noise, which lets it escape shallow local minima, and saves
computation by using only part of the data at each step.


## Stochastic gradient descent

![](figs/04-Fitting-Models/gabor-gradient-descent-paths.png)


IDEA: add noise, save computation

- Compute the gradient from only a subset of the data, a **mini-batch**
- Work through the dataset by sampling **without replacement**
- One pass through the data is an **epoch**
- Reshuffle, repeat


## Batches and epochs (30 examples, batch size 5)

```text
Data Indices  [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29]
Permute       [27 15 23 17  8  9 28 24 12  0  4 16  5 13 11 22  1  2 25  3 21 26 18 29 20  7 10 14 19  6]

Epoch # 0-----------
Step 0, Batch # 0, Batch Range [0 1 2 3 4], Batch index: [27 15 23 17 8]
Step 1, Batch # 1, Batch Range [5 6 7 8 9], Batch index: [ 9 28 24 12 0]
Step 2, Batch # 2, Batch Range [10 11 12 13 14], Batch index: [ 4 16 5 13 11]
Step 3, Batch # 3, Batch Range [15 16 17 18 19], Batch index: [22 1 2 25 3]
Step 4, Batch # 4, Batch Range [20 21 22 23 24], Batch index: [21 26 18 29 20]
Step 5, Batch # 5, Batch Range [25 26 27 28 29], Batch index: [ 7 10 14 19 6]
Epoch # 1-----------
Step 6, Batch # 0, Batch Range [0 1 2 3 4], Batch index: [27 15 23 17 8]
Step 7, Batch # 1, Batch Range [5 6 7 8 9], Batch index: [ 9 28 24 12 0]
Step 8, Batch # 2, Batch Range [10 11 12 13 14], Batch index: [ 4 16 5 13 11]
Step 9, Batch # 3, Batch Range [15 16 17 18 19], Batch index: [22 1 2 25 3]
...
```

- Batch size 5, so $30/5 = 6$ **steps** (batches) per **epoch**
- 10 epochs = 60 steps. Training budgets are quoted in either unit; LLM
  recipes quote **tokens**, which is batch size $\times$ sequence length $\times$ steps

## The SGD update

![](figs/04-Fitting-Models/gabor-gradient-descent-paths.png)


Before (full-batch gradient descent)

$$
\boldsymbol{\phi}_{t+1} \longleftarrow \boldsymbol{\phi}_t - \alpha \sum_{i=1}^{I} \frac{\partial \ell_i[\boldsymbol{\phi}_t]}{\partial \boldsymbol{\phi}},
$$

After (SGD): sum over the current batch $\mathcal{B}_t$ only

$$
\boldsymbol{\phi}_{t+1} \longleftarrow \boldsymbol{\phi}_t - \alpha \sum_{\color{red}{i \in \mathcal{B}_t}} \frac{\partial \ell_i[\boldsymbol{\phi}_t]}{\partial \boldsymbol{\phi}},
$$

Fixed learning rate $\alpha$ for now.


## Gradient descent vs. stochastic gradient descent

![](figs/04-Fitting-Models/gd-vs-sgd.png)

The cell below starts both methods at B, the start that gradient descent took
to a local minimum above. SGD uses batches of 10 of the 30 points, drawn
without replacement within an epoch, and a step size that shrinks by 1% per
step. Because the batches are random, one run proves little, so the cell also
repeats SGD with 100 different seeds and counts how often it reaches the
global minimum's valley.

In [ ]:
#| code-fold: true
#| fig-cap: "From start B, full-batch gradient descent settles in a local minimum (left); SGD with batches of 10 (seed 0) escapes it and reaches the global minimum's valley (right)."
#| fig-alt: "Two Gabor loss contour plots. Left: a smooth gradient descent path from (2.5, 8) curving right into the local minimum near (6.3, 13.5). Right: a jagged SGD path from the same start that wanders left into the central valley and ends near the global minimum at (0, 16)."
# Adapted from UDL notebook 6.3 (MIT)
def sgd(start, rng, alpha0=0.5, decay=0.99, batch_size=10, n_steps=150):
    phi = np.array(start, dtype=float)
    path = [phi]
    for t in range(n_steps):
        k = t % (len(x_gab) // batch_size)
        if k == 0:
            order = rng.permutation(len(x_gab))           # new epoch: reshuffle
        batch = order[k * batch_size:(k + 1) * batch_size]
        alpha = alpha0 * decay ** t                        # a simple schedule
        phi = phi - alpha * gabor_grad(phi, x_gab[batch], y_gab[batch])
        path.append(phi)
    return np.array(path)

gd_path = gradient_descent(starts["B"], n_steps=150)
sgd_path = sgd(starts["B"], np.random.default_rng(0))
print(f"gradient descent from B: loss = {gabor_loss(gd_path[-1]):.2f}")
print(f"SGD from B, seed 0:      loss = {gabor_loss(sgd_path[-1]):.2f}")

final = np.array([gabor_loss(sgd(starts["B"], np.random.default_rng(seed))[-1])
                  for seed in range(100)])
print(f"SGD from B, 100 seeds: {np.mean(final < 1.0):.0%} end with loss < 1 "
      f"(the global minimum's valley); median loss {np.median(final):.2f}")

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(10, 5))
plot_gabor_loss(ax0, [gd_path], "Gradient descent")
plot_gabor_loss(ax1, [sgd_path], "Stochastic gradient descent")
plt.tight_layout()
plt.show()

## Properties of SGD

- Can escape from local minima: the batch gradient is a noisy estimate of the full gradient
- Still sensible updates, since each is based on part of the data
- Uses all data equally over an epoch
- Far less computation per step, and the step count is what matters
- Finds solutions that generalize at least as well in practice
- Doesn't converge in the traditional sense: the noise keeps it moving

- So the learning rate is **decreased over time**: a **learning-rate schedule**
  (more on this later)


# Momentum

Plain (stochastic) gradient descent follows only the local slope, so in a
narrow valley it zigzags from wall to wall. A ball rolling downhill would not
do that: it carries momentum. The next optimizers add the same idea to the
gradient update.


## The zigzag problem

In [ ]:
#| code-fold: true
#| fig-alt: "Contour plot of the linear regression loss over intercept phi_0 and slope phi_1, with a green gradient descent path from (1.6, -0.5) that zigzags back and forth across the valley toward the minimum."
# Adapted from UDL notebook 6.2 (MIT); reuses lin_path from the cell above
fig, ax = plt.subplots(figsize=(5, 5))
plot_linreg_loss(ax, lin_path)
plt.show()

Think of a ball rolling down a hill.

Would it follow a path like the one on the left?

Why or why not? What is missing?

Each step is perpendicular to the contour at that point, so in an elongated
valley the steps alternate across the valley instead of along it.


## Momentum

- Keep a running **weighted average** of the gradients, and step along that
- The direction changes more slowly, and consistent components add up

$$
\begin{aligned}
\mathbf{m}_{t+1} &\leftarrow \beta \cdot \mathbf{m}_t + (1 - \beta) \sum_{\color{red}{i \in \mathcal{B}_t}} \frac{\partial \ell_i[\boldsymbol{\phi}_t]}{\partial \boldsymbol{\phi}} \\
\boldsymbol{\phi}_{t+1} &\leftarrow \boldsymbol{\phi}_t - \alpha \cdot \mathbf{m}_{t+1}
\end{aligned}
$$

- *The gradient updates the momentum.*
- *The momentum updates the weights.*
- $\beta \in [0, 1)$ controls the memory; $\beta = 0.9$ is the usual default. Still in batches.

## Without and with momentum

In [ ]:
#| code-fold: true
#| fig-alt: "Two Gabor loss contour plots over offset phi_0 and frequency phi_1, each with a green optimization path from (-1.5, 6.5). Left, without momentum: the path jumps erratically and wanders out of the central valley. Right, with momentum: the path is smoother and climbs further up the central valley."
# Adapted from UDL notebook 6.4 (MIT)
def sgd_momentum(start, rng, method, alpha=0.6, beta=0.6, batch_size=5, n_steps=81):
    phi = np.array(start, dtype=float)
    m = np.zeros(2)
    path = [phi]
    for _ in range(n_steps):
        batch = rng.permutation(len(x_gab))[:batch_size]
        if method == "sgd":
            m = gabor_grad(phi, x_gab[batch], y_gab[batch])
        elif method == "momentum":
            m = beta * m + (1 - beta) * gabor_grad(phi, x_gab[batch], y_gab[batch])
        elif method == "nesterov":   # gradient at the look-ahead point
            look_ahead = phi - alpha * beta * m
            m = beta * m + (1 - beta) * gabor_grad(look_ahead, x_gab[batch], y_gab[batch])
        phi = phi - alpha * m
        path.append(phi)
    return np.array(path)

methods = {"sgd": "Without momentum", "momentum": "With momentum",
           "nesterov": "Nesterov momentum"}
mom_paths = {k: sgd_momentum((-1.5, 6.5), np.random.default_rng(0), k) for k in methods}

fig, axes = plt.subplots(1, 2, figsize=(10, 4.4))
for ax, k in zip(axes, ["sgd", "momentum"]):
    loss = gabor_loss(mom_paths[k][-1])
    plot_gabor_loss(ax, [mom_paths[k]], f"{methods[k]}, loss = {loss:.2f}")
plt.tight_layout()
plt.show()

## Momentum on the Gabor loss

![](figs/04-Fitting-Models/momentum-paths.png)

## Nesterov accelerated momentum

- Momentum averages the gradient at the *current location* $\boldsymbol{\phi}_t$

- Nesterov: we already know the momentum will carry us about
  $\alpha \beta \mathbf{m}_t$ further, so evaluate the gradient *where we
  are about to be* (UDL eq. 6.12):

$$
\begin{aligned}
\mathbf{m}_{t+1} &\leftarrow \beta \cdot \mathbf{m}_t + (1 - \beta) \sum_{\color{red}{i \in \mathcal{B}_t}} \frac{\partial \ell_i[\boldsymbol{\phi}_t - \alpha \beta \cdot \mathbf{m}_t]}{\partial \boldsymbol{\phi}} \\
\boldsymbol{\phi}_{t+1} &\leftarrow \boldsymbol{\phi}_t - \alpha \cdot \mathbf{m}_{t+1}
\end{aligned}
$$

- The look-ahead point $\boldsymbol{\phi}_t - \alpha\beta\mathbf{m}_t$ is the
  momentum part of the next step, i.e. the update with no new gradient.
  Still in batches.


![](figs/04-Fitting-Models/nesterov-momentum.png)


The look-ahead point is $\boldsymbol{\phi}_t - \alpha\beta\mathbf{m}_t$, not
$\boldsymbol{\phi}_t - \alpha\mathbf{m}_t$: the next step's momentum term is
$\alpha\mathbf{m}_{t+1} = \alpha\beta\mathbf{m}_t + \alpha(1-\beta)(\text{new gradient})$,
and only the first part is known before we compute the new gradient. Nesterov's
original method (1983) is written for full gradients; this is Sutskever et al.'s
reformulation for mini-batch training
([Sutskever et al., 2013](https://proceedings.mlr.press/v28/sutskever13.html)).


## Nesterov momentum on the Gabor loss

In [ ]:
#| code-fold: true
#| out-width: 90%
#| fig-alt: "Three Gabor loss contour plots with green optimization paths from (-1.5, 6.5): without momentum (erratic, leaving the central valley), with momentum (smoother), and with Nesterov momentum (smoothest, ending near the global minimum)."
# Adapted from UDL notebook 6.4 (MIT); reuses mom_paths from the cell above
fig, axes = plt.subplots(1, 3, figsize=(12, 3.9))
for ax, (k, title) in zip(axes, methods.items()):
    plot_gabor_loss(ax, [mom_paths[k]], f"{title}, loss = {gabor_loss(mom_paths[k][-1]):.2f}")
plt.tight_layout()
plt.show()

for k, title in methods.items():
    finals = [gabor_loss(sgd_momentum((-1.5, 6.5), np.random.default_rng(s), k)[-1])
              for s in range(50)]
    print(f"{title:18s} median final loss over 50 seeds: {np.median(finals):.2f}")

The losses in the panel titles come from single runs with one random batch
order (seed 0): 5.60 without momentum, 2.57 with momentum and 1.05 with
Nesterov momentum. The final loss of a single run depends heavily on that
order, so the cell also prints the median over 50 seeds (3.56, 1.37 and 1.05),
which is the fairer comparison; the ordering, no momentum worst and Nesterov
best, is the same.


# Adam

Momentum smooths the path, but every parameter still uses the same step size
$\alpha$. In a deep network the gradient magnitudes of different layers differ
by orders of magnitude, so a step size that is right for one layer is wrong for
another. Adam adapts the step to each parameter.


## The challenge with fixed step sizes

![](figs/04-Fitting-Models/fixed-step-sizes.png)

**a)** $\alpha = 0.05$: moves quickly in $\phi_1$ but creeps along $\phi_0$. Too
small a step converges slowly, but eventually gets there.


**b)** $\alpha = 1.0$: too big a step moves quickly but bounces across the
valley, or diverges.


No single $\alpha$ is right for both directions.

## Solution part 1: normalized gradients

- Measure the gradient and its pointwise square

$$
\begin{aligned}
\mathbf{m}_{t+1} &\leftarrow \frac{\partial L[\boldsymbol{\phi}_t]}{\partial \boldsymbol{\phi}} \\
\mathbf{v}_{t+1} &\leftarrow \left(\frac{\partial L[\boldsymbol{\phi}_t]}{\partial \boldsymbol{\phi}}\right)^{2}
\end{aligned}
$$

- Normalize:

$$
\boldsymbol{\phi}_{t+1} \leftarrow \boldsymbol{\phi}_t - \alpha \cdot \frac{\mathbf{m}_{t+1}}{\sqrt{\mathbf{v}_{t+1}} + \epsilon}
$$

$\alpha$ is the learning rate\
$\epsilon$ is a small constant that prevents division by zero\
Square, square root and division are all pointwise


Dividing by the positive root normalizes each component to magnitude 1, so all that is left is the **sign**.


$$
\mathbf{m}_{t+1} = \begin{bmatrix} 3.0 \\ -2.0 \\ 5.0 \end{bmatrix}
\qquad
\mathbf{v}_{t+1} = \begin{bmatrix} 9.0 \\ 4.0 \\ 25.0 \end{bmatrix}
$$

$$
\frac{\mathbf{m}_{t+1}}{\sqrt{\mathbf{v}_{t+1}} + \epsilon} = \begin{bmatrix} 1.0 \\ -1.0 \\ 1.0 \end{bmatrix}
$$


## Normalized gradients: progress, but no convergence

![](figs/04-Fitting-Models/normalized-gradients.png)


- The algorithm moves downhill a fixed distance $\alpha$ along each coordinate
- Makes good progress in both directions
- But will not converge unless it happens to land exactly on the minimum: the
  step never gets smaller


## Adaptive moment estimation (Adam)

- Compute the mean and the pointwise squared gradient *with momentum*
  ([Kingma & Ba, 2015](https://arxiv.org/abs/1412.6980))

$$
\begin{aligned}
\mathbf{m}_{t+1} &\leftarrow \beta \cdot \mathbf{m}_t + (1 - \beta) \frac{\partial L[\boldsymbol{\phi}_t]}{\partial \boldsymbol{\phi}} \\
\mathbf{v}_{t+1} &\leftarrow \gamma \cdot \mathbf{v}_t + (1 - \gamma) \left(\frac{\partial L[\boldsymbol{\phi}_t]}{\partial \boldsymbol{\phi}}\right)^2
\end{aligned}
$$

- **Bias correction**: both start at zero, so the weights on past gradients sum
  to $1 - \beta^{t+1}$ and $1 - \gamma^{t+1}$, not 1: early averages are too
  small. Divide it out:

$$
\begin{aligned}
\tilde{\mathbf{m}}_{t+1} &\leftarrow \frac{\mathbf{m}_{t+1}}{1 - \beta^{t+1}} & \qquad \mathbf{m}_{0} &= \mathbf{0} \\
\tilde{\mathbf{v}}_{t+1} &\leftarrow \frac{\mathbf{v}_{t+1}}{1 - \gamma^{t+1}} & \qquad \mathbf{v}_{0} &= \mathbf{0}
\end{aligned}
$$

- Update the parameters with the normalized, smoothed gradient

$$
\boldsymbol{\phi}_{t+1} \leftarrow \boldsymbol{\phi}_t - \alpha \cdot \frac{\tilde{\mathbf{m}}_{t+1}}{\sqrt{\tilde{\mathbf{v}}_{t+1}} + \epsilon}
$$


Where the bias correction comes from: after $t+1$ steps from
$\mathbf{m}_0 = \mathbf{0}$, the weights on the gradients seen so far are
$(1-\beta), (1-\beta)\beta, \ldots, (1-\beta)\beta^{t}$, which sum to
$1 - \beta^{t+1}$ rather than 1. If every gradient were the same vector
$\mathbf{g}$, $\mathbf{m}_{t+1}$ would equal $(1 - \beta^{t+1})\mathbf{g}$, so
dividing by $1 - \beta^{t+1}$ restores $\mathbf{g}$. With $\gamma = 0.999$ the
correction for $\mathbf{v}$ matters for hundreds of steps: at $t+1 = 100$,
$1 - 0.999^{100} \approx 0.0952$, so the uncorrected $\mathbf{v}$ would be ten
times too small and the step, which divides by $\sqrt{\mathbf{v}}$, about
3.2$\times$ too large ($1/\sqrt{0.0952} \approx 3.24$).

The cell below implements Adam in a dozen lines on the two-parameter loss of
UDL notebook 6.5, with $\beta = 0.9$ and PyTorch's default $\gamma = 0.999$,
and runs it with and without the bias correction. It prints the length of the
first steps, then the factor $1/\sqrt{1-\gamma^{t}}$ and the net factor
$(1-\beta^{t})/\sqrt{1-\gamma^{t}}$ by which skipping both corrections
inflates the step after $t$ updates. By $t = 100$, $\beta^t$ is negligible and
the two agree at 3.24.

In [ ]:
#| code-fold: true
#| fig-cap: "Adam ($\\alpha = 0.05$, $\\beta = 0.9$, $\\gamma = 0.999$, 60 steps) with bias correction (left) and without it (right). Without it the early steps are about three times too long and the path overshoots the valley."
#| fig-alt: "Two contour plots of a loss with a minimum at phi_0 = 0.7, phi_1 = 0. Left, with bias correction: evenly spaced steps from (-0.7, -0.9) curve smoothly into the minimum. Right, without bias correction: the first steps are much longer and the path overshoots across the valley before settling."
# Adapted from UDL notebook 6.5 (MIT)
def bowl_loss(phi):
    return 1.0 - np.exp(-2.0 * phi[1] ** 2 - (phi[0] - 0.7) ** 2 / 8.0)

def bowl_grad(phi):
    height = 1.0 - bowl_loss(phi)
    return np.array([height * (phi[0] - 0.7) / 4.0, height * 4.0 * phi[1]])

def adam(start, alpha=0.05, beta=0.9, gamma=0.999, eps=1e-8, n_steps=60,
         bias_correction=True):
    phi = np.array(start, dtype=float)
    m, v = np.zeros(2), np.zeros(2)
    path = [phi]
    for t in range(n_steps):
        g = bowl_grad(phi)
        m = beta * m + (1 - beta) * g
        v = gamma * v + (1 - gamma) * g ** 2
        if bias_correction:
            m_hat, v_hat = m / (1 - beta ** (t + 1)), v / (1 - gamma ** (t + 1))
        else:
            m_hat, v_hat = m, v
        phi = phi - alpha * m_hat / (np.sqrt(v_hat) + eps)
        path.append(phi)
    return np.array(path)

adam_paths = {bc: adam((-0.7, -0.9), bias_correction=bc) for bc in (True, False)}
for bc, path in adam_paths.items():
    steps = np.linalg.norm(np.diff(path, axis=0), axis=1)
    print(f"bias correction {'on ' if bc else 'off'}: first step lengths {np.round(steps[:4], 3)}")

# Skipping the correction shrinks m by (1 - beta^t) and sqrt(v) by sqrt(1 - gamma^t)
beta, gamma = 0.9, 0.999
for t in [1, 10, 100, 1000]:
    print(f"t = {t:4d}: 1/sqrt(1 - gamma^t) = {1 / np.sqrt(1 - gamma ** t):5.2f}, "
          f"uncorrected step / corrected step = {(1 - beta ** t) / np.sqrt(1 - gamma ** t):5.2f}")

g0, g1 = np.meshgrid(np.linspace(-1, 1.5, 251), np.linspace(-1, 1, 201))
bowl = bowl_loss(np.array([g0, g1]))
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for ax, bc in zip(axes, (True, False)):
    ax.contourf(g0, g1, bowl, 256, cmap=udl_cmap)
    ax.contour(g0, g1, bowl, 20, colors="#80808080", linewidths=0.8)
    ax.plot(*adam_paths[bc].T, ".-", color="#a0d9d3", ms=6)
    ax.set_title(f"Adam, bias correction {'on' if bc else 'off'}", fontsize=10)
    ax.set_xlabel(r"$\phi_0$"); ax.set_ylabel(r"$\phi_1$")
plt.tight_layout()
plt.show()

## Adam converges where normalized gradients did not

![](figs/04-Fitting-Models/normalized-gradients-vs-adam.png)

The momentum in $\mathbf{m}$ averages out the sign flips near the minimum, so
the effective step shrinks as the path settles.

## Why Adam is the default

- Gradients can shrink or grow with depth in a network; Adam's per-parameter
  normalization balances the effective step across layers
- Less sensitive to the initial learning rate than SGD, so it needs less tuning
- The UDL text adds that it "doesn't need complex learning rate schedules".
  In practice **every** large training run still pairs Adam with a schedule,
  and the next section explains why

# Learning-rate schedules

Momentum and Adam fix the *direction* and the *relative* size of each step. The
overall scale $\alpha$ is still a free choice, and it turns out the best choice
changes over the course of training. A learning-rate schedule is the rule
$\alpha_t$ that sets it at each step.


## Why change the learning rate at all?

- SGD's noise scales with $\alpha$: a large step explores and escapes bad
  regions early, a small step is needed to settle into a minimum at the end
- Classic answer: **step decay** (divide by 10 every $k$ epochs) or
  $\alpha_t \propto 1/t$

- **Cosine annealing** ([Loshchilov & Hutter, 2017, SGDR](https://arxiv.org/abs/1608.03983)):
  smooth decay from $\alpha_{\max}$ to $\alpha_{\min}$ over $T$ steps

$$
\alpha_t = \alpha_{\min} + \tfrac{1}{2}\left(\alpha_{\max} - \alpha_{\min}\right)\left(1 + \cos\left(\pi \frac{t}{T}\right)\right)
$$

- SGDR restarts the cosine (periods $T, 2T, 4T, \ldots$) to re-explore; the
  single cosine without restarts is what stuck


- Modern recipes add a **warmup**: ramp $\alpha_t$ linearly from 0 to
  $\alpha_{\max}$ over the first few thousand steps
  ([Goyal et al., 2017](https://arxiv.org/abs/1706.02677))


## Three schedules used today

In [ ]:
#| code-fold: true
#| out-width: 85%
#| fig-cap: "Learning rate as a fraction of its peak against the fraction of training steps. a) Linear warmup then cosine decay to 10% of the peak, as in [GPT-3](https://arxiv.org/abs/2005.14165) and [Llama 2](https://arxiv.org/abs/2307.09288). b) Cosine annealing with warm restarts ([Loshchilov & Hutter, 2017](https://arxiv.org/abs/1608.03983)). c) Warmup-stable-decay ([Hu et al., 2024, MiniCPM](https://arxiv.org/abs/2404.06395))."
#| fig-alt: "Three line plots of learning rate over the fraction of training steps. a) A linear ramp from 0 to the peak over the first 5%, then a cosine curve decaying to 10% of the peak. b) Three cosine decays to zero with instant jumps back to the peak, with periods of 1/7, 2/7 and 4/7 of training. c) A linear ramp to the peak, a flat line at the peak until 80%, then a linear cooldown to zero."
T = 1000                    # total training steps
t = np.arange(T + 1)

def warmup_cosine(t, warmup=50, floor=0.1):
    progress = np.clip((t - warmup) / (T - warmup), 0.0, 1.0)
    cosine = floor + 0.5 * (1.0 - floor) * (1.0 + np.cos(np.pi * progress))
    return np.where(t < warmup, t / warmup, cosine)

def sgdr(t, t0=T / 7, t_mult=2):
    """Cosine annealing with warm restarts: periods T0, 2 T0, 4 T0 (they sum to T)."""
    lr = np.empty(len(t))
    start, period = 0.0, t0
    for i, step in enumerate(t):
        while step >= start + period:
            start, period = start + period, period * t_mult
        lr[i] = 0.5 * (1.0 + np.cos(np.pi * (step - start) / period))
    return lr

def warmup_stable_decay(t, warmup=50, decay_start=800):
    lr = np.ones(len(t))
    lr[t < warmup] = t[t < warmup] / warmup
    cooldown = t >= decay_start
    lr[cooldown] = 1.0 - (t[cooldown] - decay_start) / (T - decay_start)
    return lr

panels = [("a) Warmup + cosine decay to 10%", warmup_cosine(t), "#2a78d6"),
          ("b) Cosine with warm restarts (SGDR)", sgdr(t), "#eb6834"),
          ("c) Warmup-stable-decay (WSD)", warmup_stable_decay(t), "#1baf7a")]

fig, axes = plt.subplots(1, 3, figsize=(12, 3.2), sharey=True, constrained_layout=True)
for ax, (title, lr, color) in zip(axes, panels):
    ax.plot(t / T, lr, color=color, linewidth=2.2)
    ax.axhline(1.0, color="#d9d8d4", linestyle="--", zorder=0)
    ax.set_title(title, fontsize=11, loc="left")
    ax.set_xlabel("Fraction of training steps")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.08)
    ax.grid(True, color="#d9d8d4", linewidth=0.8)
    ax.spines[["top", "right"]].set_visible(False)
axes[0].set_ylabel(r"Learning rate $\alpha_t$ / peak")
plt.show()

- **Warmup + cosine** (a): the standard LLM pretraining schedule since GPT-3
- **Warmup-stable-decay, WSD** (c): hold $\alpha_{\max}$, then a short cooldown
  (10–20% of steps). The flat phase can be stopped and cooled down at *any*
  length, so one run gives checkpoints for many training budgets
  ([Hägele et al., 2024](https://arxiv.org/abs/2405.18392)). DeepSeek-V3 uses
  a warmup–stable–decay-style schedule whose decay phase is a cosine
  ([DeepSeek-AI, 2024, §4.2](https://arxiv.org/abs/2412.19437))

## Why warmup matters, especially for Adam

- Adam's step is $\alpha\,\tilde{\mathbf{m}}/\sqrt{\tilde{\mathbf{v}}}$.
  Bias correction fixes the *expected size* of $\tilde{\mathbf{v}}$ at the start,
  but not its **variance**: after a handful of steps $\tilde{\mathbf{v}}$ is an
  average of a handful of squared gradients

- A coordinate whose first few gradients happened to be tiny gets a huge
  normalized step. Warmup keeps $\alpha_t$ small until $\tilde{\mathbf{v}}$ has
  seen enough samples ([Liu et al., 2020, RAdam](https://arxiv.org/abs/1908.03265),
  who show warmup acts as a variance reducer)


- Independently, the loss is **sharpest** near initialization. Small early
  steps let the parameters move into a flatter region before full-size steps
  are taken ([Gilmer et al., 2022](https://arxiv.org/abs/2110.04369))


- [Wortsman et al., 2023](https://arxiv.org/abs/2309.14322) study how warmup
  length and learning rate interact with training **instabilities**


RAdam ([Liu et al., 2020](https://arxiv.org/abs/1908.03265)) shows the
adaptive step's variance is large in the first steps and proposes a
rectification; warmup is the simple fix. In practice warmups run for hundreds
to thousands of steps (GPT-3 warms up over the first 375M tokens, Llama 2 over
2,000 steps, Llama 3 over 8,000 steps). Using a smaller $\gamma$ such as 0.95,
as the LLM recipes do, makes $\tilde{\mathbf{v}}$ track changes in gradient
scale faster.


# Weight decay and AdamW

The last ingredient is regularization. The textbook treats L2 regularization
and weight decay as the same thing, and for plain SGD they are. For Adam they
are not, and the difference is why every modern recipe says "AdamW" rather
than "Adam".


## L2 regularization with SGD is weight decay

Add a penalty on the size of the parameters to the loss:

$$
\tilde{L}[\boldsymbol{\phi}] = L[\boldsymbol{\phi}] + \frac{\lambda}{2}\,\|\boldsymbol{\phi}\|^2
\qquad\Longrightarrow\qquad
\frac{\partial \tilde{L}}{\partial \boldsymbol{\phi}} = \frac{\partial L}{\partial \boldsymbol{\phi}} + \lambda \boldsymbol{\phi}
$$

Plug it into the gradient descent step and collect the $\boldsymbol{\phi}_t$ terms:

$$
\boldsymbol{\phi}_{t+1} \leftarrow \boldsymbol{\phi}_t - \alpha\left(\frac{\partial L}{\partial \boldsymbol{\phi}} + \lambda \boldsymbol{\phi}_t\right)
= (1 - \alpha\lambda)\,\boldsymbol{\phi}_t - \alpha \frac{\partial L}{\partial \boldsymbol{\phi}}
$$

Each step first **shrinks every weight** by the factor $(1 - \alpha\lambda)$,
then takes the ordinary gradient step. That shrinkage is **weight decay**.


For SGD the two views are the same update. With momentum they already differ
slightly; with Adam they differ a lot.


## Adam + L2 vs. AdamW: the two updates side by side

**Adam with L2 penalty** (the penalty gradient goes *through* the moments)

$$
\begin{aligned}
\mathbf{g}_t &= \frac{\partial L}{\partial \boldsymbol{\phi}} + {\color{red}\lambda \boldsymbol{\phi}_t} \\
\mathbf{m}_{t+1} &\leftarrow \beta \mathbf{m}_t + (1-\beta)\,\mathbf{g}_t \\
\mathbf{v}_{t+1} &\leftarrow \gamma \mathbf{v}_t + (1-\gamma)\,\mathbf{g}_t^2 \\
\boldsymbol{\phi}_{t+1} &\leftarrow \boldsymbol{\phi}_t - \alpha \frac{\tilde{\mathbf{m}}_{t+1}}{\sqrt{\tilde{\mathbf{v}}_{t+1}} + \epsilon}
\end{aligned}
$$


**AdamW: decoupled weight decay**
([Loshchilov & Hutter, 2019](https://arxiv.org/abs/1711.05101))

$$
\begin{aligned}
\mathbf{g}_t &= \frac{\partial L}{\partial \boldsymbol{\phi}} \\
\mathbf{m}_{t+1} &\leftarrow \beta \mathbf{m}_t + (1-\beta)\,\mathbf{g}_t \\
\mathbf{v}_{t+1} &\leftarrow \gamma \mathbf{v}_t + (1-\gamma)\,\mathbf{g}_t^2 \\
\boldsymbol{\phi}_{t+1} &\leftarrow \boldsymbol{\phi}_t - \alpha \left(\frac{\tilde{\mathbf{m}}_{t+1}}{\sqrt{\tilde{\mathbf{v}}_{t+1}} + \epsilon} + {\color{red}\lambda \boldsymbol{\phi}_t}\right)
\end{aligned}
$$


- Left: the decay term $\lambda\boldsymbol{\phi}_t$ is divided by
  $\sqrt{\tilde{\mathbf{v}}}$ like everything else, so weights with large
  historical gradients are **barely regularized**, and $\lambda$ and $\alpha$
  interact
- Right: every weight decays at the same rate $\alpha\lambda$, exactly as in the
  SGD derivation, and the best $\lambda$ no longer depends on $\alpha$


Loshchilov & Hutter showed empirically that Adam with an L2 penalty
generalized worse than SGD with momentum on image classification, and that the
decoupled form closed the gap. In the decoupled update both the gradient step
and the decay are multiplied by the schedule $\alpha_t$, so weight decay
switches off at the same rate as learning does. PyTorch's `torch.optim.AdamW`
implements exactly the right-hand column, with the decay applied as
$\boldsymbol{\phi} \leftarrow (1 - \alpha\lambda)\boldsymbol{\phi}$ before the
Adam step; `torch.optim.Adam(weight_decay=...)` implements the left-hand
column. The "weight decay 0.1" in LLM recipes refers to $\lambda$ in this
PyTorch convention.

A small numeric check of the claim. Two weights start at 1.0 and receive
zero-mean gradients that alternate in sign: weight 1 with magnitude 10 (a
"loud" weight), weight 2 with magnitude 0.01 (a "quiet" one). The cell runs
both updates for 20 steps with $\alpha = 0.01$ and $\lambda = 0.1$ and reports
how much the decay term shrank each weight, measured against the same run with
$\lambda = 0$.

In [ ]:
alpha, lam, beta, gamma, eps = 0.01, 0.1, 0.9, 0.999, 1e-8
grad_scale = np.array([10.0, 0.01])          # weight 1 loud, weight 2 quiet

def adam_decay_run(mode, lam, n_steps=20):
    phi, m, v = np.ones(2), np.zeros(2), np.zeros(2)
    for t in range(n_steps):
        g = grad_scale * (-1) ** t           # zero-mean loss gradient
        if mode == "Adam + L2":
            g = g + lam * phi                # penalty gradient goes through m and v
        m = beta * m + (1 - beta) * g
        v = gamma * v + (1 - gamma) * g ** 2
        m_hat, v_hat = m / (1 - beta ** (t + 1)), v / (1 - gamma ** (t + 1))
        if mode == "AdamW":
            phi = phi - alpha * lam * phi    # decoupled: same rate for every weight
        phi = phi - alpha * m_hat / (np.sqrt(v_hat) + eps)
    return phi

no_decay = adam_decay_run("Adam + L2", lam=0.0)
for mode in ["Adam + L2", "AdamW"]:
    shrink = no_decay - adam_decay_run(mode, lam)
    print(f"{mode:9s}: shrinkage from decay  weight 1 (loud) = {shrink[0]:.4f}, "
          f"weight 2 (quiet) = {shrink[1]:.4f}")
print(f"SGD-style weight decay, 1 - (1 - alpha*lam)^20 = {1 - (1 - alpha * lam) ** 20:.4f}")

With the L2 penalty the loud weight is barely decayed and the quiet one is
decayed far more than $\lambda$ asks for, because the penalty is divided by each
weight's own $\sqrt{\tilde{\mathbf{v}}}$. AdamW decays both by the same
amount, essentially the $1-(1-\alpha\lambda)^{20}$ of plain SGD.


# What a 2026 recipe uses

Everything above was published between 2013 and 2019, and it is still, with
remarkably few changes, what trains the largest open-weight models. One slide
of numbers, with their sources, and one on the first serious alternative to
Adam in a decade.


## The LLM pretraining recipe, line by line

| Setting | Value | What it does |
|---|---|---|
| Optimizer | AdamW | Adam with decoupled weight decay |
| $\beta_1, \beta_2$ (UDL's $\beta, \gamma$) | 0.9, 0.95 | $\beta_2 = 0.95$ rather than 0.999: a shorter memory for $\mathbf{v}$, so it tracks gradient-scale changes faster and spikes less |
| Weight decay $\lambda$ | 0.1 | Applied to the weight matrices; usually not to biases or normalization gains |
| Schedule | linear warmup, then cosine decay to 10% of peak | Warmup 2,000 steps (Llama 2) |
| Peak learning rate | about $3 \times 10^{-4}$ (7B) down to $1.5 \times 10^{-4}$ (70B) | Larger models take smaller steps |
| Gradient clipping | global norm 1.0 | If $\|\mathbf{g}\| > 1$, rescale $\mathbf{g} \leftarrow \mathbf{g}/\|\mathbf{g}\|$ before the update; bounds the damage of one bad batch ([Pascanu et al., 2013](https://arxiv.org/abs/1211.5063)) |

Sources: [Llama 2, Touvron et al. 2023, §2.2](https://arxiv.org/abs/2307.09288),
which copies [GPT-3, Brown et al. 2020, App. B](https://arxiv.org/abs/2005.14165).
[Llama 3 (2024)](https://arxiv.org/abs/2407.21783) keeps AdamW with warmup
(8,000 steps) + cosine; [DeepSeek-V3 (2024)](https://arxiv.org/abs/2412.19437)
keeps $\beta = (0.9, 0.95)$, $\lambda = 0.1$, clipping 1.0 and uses a
warmup–stable–decay-style schedule whose decay phase is a cosine (§4.2).

Two things are worth noticing. First, the numbers are inherited: Llama 2's
$\beta_2 = 0.95$, weight decay 0.1, clip 1.0 and "cosine to 10%" are GPT-3's,
and DeepSeek-V3 uses the same $\beta$, decay and clipping.
Second, what *has* changed since 2020 is mostly the schedule (WSD) and the
batch size, which modern runs ramp up over the first part of training. The
reason for clipping at the global norm rather than per element is that it
preserves the direction of the update: a rare, badly scaled batch is shortened
rather than redirected.


## Muon: a newer alternative

- Adam treats every parameter as an independent scalar. **Muon** treats each
  weight *matrix* as a unit ([Jordan et al., 2024](https://kellerjordan.github.io/posts/muon/))
- Idea: take the momentum-SGD update matrix $\mathbf{G}$, and replace it with
  the nearest **orthogonal** matrix (all singular values set to 1) before
  applying it. In symbols, if $\mathbf{G} = \mathbf{U}\mathbf{S}\mathbf{V}^\top$,
  step along $\mathbf{U}\mathbf{V}^\top$, computed cheaply on the GPU by a few
  Newton–Schulz iterations rather than an SVD
- Effect: rare directions in the update get the same weight as common ones,
  which Adam's per-element normalization cannot do
- Used only for the hidden-layer matrices; embeddings, output head and
  scalars still use AdamW

- Scaled up by Moonshot AI: *Muon is Scalable for LLM Training*
  ([Liu et al., 2025](https://arxiv.org/abs/2502.16982)) adds weight decay and
  reports about 2$\times$ the compute efficiency of AdamW; **Kimi K2** (1T
  parameters) was trained with it plus a stability fix, "MuonClip"
  ([Moonshot AI, 2025](https://arxiv.org/abs/2507.20534))
- Karpathy's [nanochat](https://github.com/karpathy/nanochat) (2025) uses Muon
  for the matrices and AdamW for the rest, in about a hundred lines you can read


We do not derive Muon here. The one idea to keep is that Adam, AdamW and Muon
all answer the same question, "how should the raw gradient be reshaped before
it becomes a step?", with different normalizations: Adam per element, Muon per
matrix. Whether Muon displaces AdamW is an open question as of 2026:
[DeepSeek-V3](https://arxiv.org/abs/2412.19437) uses AdamW;
[Qwen3](https://arxiv.org/abs/2505.09388) and
[Llama 4](https://ai.meta.com/blog/llama-4-multimodal-intelligence/) do not
state their optimizer; [Kimi K2](https://arxiv.org/abs/2507.20534) and
[GLM-4.5](https://arxiv.org/abs/2508.06471) (2025) use Muon.


## Hyperparameters you now own

- **Optimizer**: SGD, SGD + (Nesterov) momentum, Adam, AdamW, Muon
- **Learning rate**: peak value, warmup length, decay shape (cosine or WSD) and
  final value
- **Momentum coefficients**: $\beta$ (and $\gamma$ for Adam)
- **Weight decay** $\lambda$, and which parameters it applies to
- **Gradient clipping** threshold
- **Batch size**, which sets how noisy each gradient is and interacts with all
  of the above

Defaults that work: AdamW, $\beta = (0.9, 0.95)$ or $(0.9, 0.999)$ for small
models, $\lambda = 0.1$, warmup + cosine, clip at 1.0. Tune the peak learning
rate first; it matters most.


# Summary

## Recap

- **Gradient descent** – Follow the negative gradient to a minimum; on a
  non-convex loss which minimum depends on the start
- **Stochastic gradient descent** – Compute gradients on mini-batches: less
  compute per step, and noise that helps escape local minima
- **(Nesterov) momentum** – Average gradients over steps to smooth the path;
  Nesterov evaluates the gradient at the look-ahead point
  $\boldsymbol{\phi}_t - \alpha\beta\mathbf{m}_t$
- **Adam** – Normalize each parameter's step by its running RMS gradient, with
  bias correction for the zero start
- **Learning-rate schedules** – Warmup (protects Adam's noisy early
  $\tilde{\mathbf{v}}$), then cosine decay or warmup-stable-decay
- **Weight decay vs. L2** – Identical for SGD; for Adam use the decoupled form,
  AdamW
- **The 2026 recipe** – AdamW $\beta = (0.9, 0.95)$, decay 0.1, warmup + cosine,
  clip at 1.0; Muon is the newer matrix-aware alternative

## Next

- Where the gradient comes from: the chain rule, backpropagation in matrix
  form, computational graphs and autograd
- Initialization: why the starting point of gradient descent matters in deep
  networks
- Then: generalization and regularization, and convolutional networks